Instructions<br>
• pip install datasets transformers evaluate jiwer<br>
• import torch, import datasets, import evaluate, import numpy as np, from dataclasses import data class, field. from typing import Any, Dict, List, Optional, Union<br>
• load train and test splits of Common Voice Scripted Speech 25.0 - Assamese.tar.gz  dataset (https://drive.google.com/file/d/17I3oHLlGDwqWCfB3U0OBpxsh0xYN1VeX/view?usp=drive_link)
• for test, take only first 10 examples and proceed (otherwise you will hit Out of Memory Error)<br>
• from transformers import WhisperFeatureExtractor<br>
• Load feature extractor from the pretrained whisper tiny of openai<br>
• from transformers import WhisperTokenizer<br>
• load tokenizer of openai whisper tiny for assamese for asr task<br>
• from transformers import WhisperProcessor<br>
• load processor of openai whisper tiny for assamese for asr task<br>
• from datasets import Audio<br>
• convert audio to sampling rate of 16k<br>
• write a function which takes batches of input data and gives batches with features    extracted and corresponding labels from the tokenizer.<br>
• from transformers import WhisperForConditionalGeneration<br>
• model = WhisperForConditionalGeneration.from pretrained(”openai/whisper-tiny”)<br>
• set language to assamese, task to asr and forced decoder ids to none<br>
• Write a data collator<br>
• import evaluate and set metric = evaluate.load(”wer”)<br>
• write a function def compute metrics(pred) which takes predictions and returns wer<br>
• from transformers import Seq2SeqTrainingArguments<br>
• set the arguments for Seq2SeqTrainingArguments() as mentioned in the link
“https://huggingface.co/blog/fine-tune-whisper” but change warmup steps to 50, max steps to 100, per device train batch size=8, per device eval batch size=1, save steps=100, eval steps=100<br>
• from transformers import Seq2SeqTrainer<br>
• Set the arguments based on the info mentioned in the above link<br>
• start training<br>

In [3]:
!pip install datasets transformers evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.8 MB/s eta 0:00:00


In [4]:
import torch
import datasets
import evaluate
import numpy as np

from dataclasses import dataclass, field
from typing import Any, Dict, Optional, Union
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
from datasets import Audio
from transformers import WhisperForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

In [3]:
!tar -xvzf "/content/Common Voice Scripted Speech 25.0 - Assamese(1).tar.gz"

cv-corpus-25.0-2026-03-09/as/README.md
cv-corpus-25.0-2026-03-09/as/clip_durations.tsv
cv-corpus-25.0-2026-03-09/as/clips/
cv-corpus-25.0-2026-03-09/as/dev.tsv
cv-corpus-25.0-2026-03-09/as/invalidated.tsv
cv-corpus-25.0-2026-03-09/as/other.tsv
cv-corpus-25.0-2026-03-09/as/reported.tsv
cv-corpus-25.0-2026-03-09/as/test.tsv
cv-corpus-25.0-2026-03-09/as/train.tsv
cv-corpus-25.0-2026-03-09/as/unvalidated_sentences.tsv
cv-corpus-25.0-2026-03-09/as/validated.tsv
cv-corpus-25.0-2026-03-09/as/validated_sentences.tsv
cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20870771.mp3
cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20870772.mp3
cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20870773.mp3
cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20870774.mp3
cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20870775.mp3
cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20878095.mp3
cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20878096.mp3
cv-corpus-25.0-2026-03-09/as/clips/com

In [4]:
import pandas as pd
df_tsv = pd.read_csv("/content/cv-corpus-25.0-2026-03-09/as/train.tsv", sep="\t")

In [5]:
df_test = pd.read_csv("/content/cv-corpus-25.0-2026-03-09/as/test.tsv", sep="\t", nrows=10)

In [ ]:
df_tsv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 956 entries, 0 to 955
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   client_id        956 non-null    object 
 1   path             956 non-null    object 
 2   sentence_id      956 non-null    object 
 3   sentence         956 non-null    object 
 4   sentence_domain  0 non-null      float64
 5   up_votes         956 non-null    int64  
 6   down_votes       956 non-null    int64  
 7   age              172 non-null    object 
 8   gender           172 non-null    object 
 9   accents          686 non-null    object 
 10  variant          0 non-null      float64
 11  locale           956 non-null    object 
 12  segment          0 non-null      float64
dtypes: float64(3), int64(2), object(8)
memory usage: 97.2+ KB


In [ ]:
df_test.iloc[0]

,0
client_id,4d49bfe22b9c07fce9de4613cb2b50506a859d8e2f95ad...
path,common_voice_as_25111102.mp3
sentence_id,43d0ec67fb1abb2cf8fb92bf2020bcb1a646d3641cf5cf...
sentence,আমাৰ ঘৰত তলা মৰা অৱস্থাত ওছৰৰে ঘৰত মানুহ নথকা ...
sentence_domain,NaN
up_votes,2
down_votes,0
age,NaN
gender,NaN
accents,NaN


In [ ]:
from IPython.display import Audio
Audio("/content/cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20878096.mp3")

# Q.No - 1

How many examples are present in the train split of Common Voice Scripted Speech 25.0 - Assamese language dataset ?

In [6]:
df_tsv.shape[0]

956

## Q.No - 2
How many unique characters are there in the train split text ?

In [ ]:
unique_char = {}

for i in df_tsv["sentence"]:
  for j in i:
    if j not in unique_char:
      unique_char[j] = 1
    else:
      unique_char[j] += 1

In [ ]:
print("Total unique charcters in the train dataset",len(unique_char))

Total unique charcters in the train dataset 74


## Q.No - 3
What is the sampling rate of the original Common Voice Scripted Speech 25.0 - Assamese language audio in Hz?

In [13]:
import torchaudio

waveform, sampling_rate = torchaudio.load("/content/cv-corpus-25.0-2026-03-09/as/clips/common_voice_as_20878096.mp3")
print("The sampling rate of the language is ", sampling_rate)

The sampling rate of the language is  48000


## Q.No - 4
What is the format of the Common Voice Scripted Speech 25.0 - Assamese language audio ?

Ans :
From the previoius question itself we can notice that, we had loaded the .mp3 file. So the answer is `.mp3`

## Q.No - 5
What will be the window length in msec if n fft is 400 in "WhisperFeatureExtractor" ?

In [ ]:
# Parameters
n_fft = 400
sampling_rate = 16000  # Whisper default

# Calculate window length in ms
window_length_ms = (n_fft / sampling_rate) * 1000

print("Window length:", window_length_ms, "ms")

Window length: 25.0 ms


## Q.No - 6
What is the first token number after tokenising the 56th example?

In [5]:
# Load processor (feature extractor + tokenizer bundled)
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="Assamese", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="Assamese", task="transcribe")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

In [8]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model.config.forced_decoder_ids = None
model.config.language = "Assamese"
model.config.task = "transcribe"

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

In [14]:
def prepare_dataset(batch):
  array, sampling_rate = torchaudio.load("/content/cv-corpus-25.0-2026-03-09/as/clips/"+batch["path"])
  resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
  waveform_16k = resampler(waveform)
  inputs = feature_extractor(waveform_16k.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
  batch["input_features"] = inputs.input_features[0]
  batch["labels"] = tokenizer(batch["sentence"]).input_ids
  print(tokenizer(batch["sentence"]).input_ids[0])
  #return batch

In [9]:
from datasets import Dataset
train = Dataset.from_pandas(df_tsv)
test = Dataset.from_pandas(df_test)

In [15]:
print("The token number is ",prepare_dataset(train[56]))

50258
The token number is  None


## Q.No - 7
What is the token corresponding to the token number 51833 in whisper?

In [7]:
# Get the string form directly
print(tokenizer.convert_ids_to_tokens([51833]))

['<|29.38|>']


## Q.No - 8
Is token number 51833 a special token?

In [10]:
#This will not work because, .decode does post process work and gives finisehd senence. Means, it romes the special tokens.
#tokenizer.decode([51833])
if 51833  in tokenizer.all_special_ids:
  print("Yes")
else:
  print("No")

No


## Q.No - 9
What is the token corresponding to the token number 50350 in whisper?

In [18]:
tokenizer.decode([50350])

'<|as|>'

## Q.No - 10
Is token number 50350 a special token?

In [11]:
if 50350  in tokenizer.all_special_ids:
  print("Yes")
else:
  print("No")

Yes
